# 🧩 OOP in Python — Part 3: A Multi-File Mini Project (Movie Reservation System)

Parts 1 and 2 kept every class in a single notebook cell. Real projects split classes across **multiple files** (modules), each with one responsibility. This notebook documents a small **Movie Ticket Reservation System** originally written as 4 separate `.py` files, showing how `main.py`, `movie.py`, `customer.py`, and `booking.py` work together.

## 🎯 Learning Goal

By the end of this notebook you should understand:
- How a project is typically split into a **driver script** (`main.py`) and supporting **modules** (`movie.py`, `customer.py`, `booking.py`)
- How **encapsulation** looks when data + the methods that use it are grouped into one class (`MovieReservation`)
- How a **class variable** (`Booking.booking_counter`) can generate unique, ever-increasing IDs shared across every booking
- Why code split across real files needs **imports** to work, and why copy-pasting that code into one notebook cell needs those imports removed

## 🤔 What is a Multi-File Project? (Real-life analogy)

Think of a restaurant kitchen 🍳. One station preps vegetables, another grills, another plates the dish, and a head chef (the driver script) coordinates them all — nobody tries to do everything at one station. In code, `movie.py` "preps" reservation logic, `customer.py` "preps" customer data, `booking.py` "preps" ID generation, and `main.py` is the head chef that calls on each station as needed via `input()`-driven menu choices.

### 🧠 Key Idea

- Splitting a program into multiple files keeps each class focused on **one responsibility** — `Customer` only knows about a customer's name, `Booking` only knows how to generate IDs, `MovieReservation` coordinates the actual seat logic.
- Modules that depend on each other need `import` statements (e.g. `movie.py` imports `Booking` and `Customer`) — this only works when the files are laid out as an actual Python package on disk, not when the code is pasted into a notebook.
- A **class variable** like `Booking.booking_counter` persists and increments across every object created — it's what guarantees Booking IDs never repeat.
- `main.py` (the "driver program") is intentionally kept thin — it just creates objects and calls their methods in a loop. All the real logic (validation, state changes) stays inside the classes.

### 📚 Important Terms

| Term | Simple Meaning | Example |
|---|---|---|
| Module | A single `.py` file containing related code | `movie.py`, `customer.py` |
| Driver Program | The script that creates objects and drives the program's flow | `main.py` |
| Package | A folder of modules that can import each other | `oops/book_reservation/` |
| `@classmethod` | A method bound to the class itself (via `cls`), not an instance | `Booking.generate_booking_id()` |
| Class Variable | A value shared by every object/call, used here as a running counter | `Booking.booking_counter` |
| Separation of Concerns | Design principle: each class/file handles one job only | `Customer` only stores a name |

## 1. 📄 `customer.py` — The `Customer` Class

The simplest of the four files — pure data storage. It only exists to bundle a customer's name (encapsulation in its most minimal form: a `Customer` object always carries its own name with it, so no other code needs to pass names around as loose strings).

In [ ]:
"""
===========================================================
File Name : customer.py

THEORY - CLASS AND OBJECT

Class:
------
A class is a blueprint or template used to create objects.
It defines the properties (attributes) and behaviors (methods)
that an object will have.

Object:
-------
An object is a real-world instance of a class.

Example:
Car -> Class
BMW -> Object

Here,
Customer is a class.
Every customer who books a ticket becomes an object of this class.

OOP Concepts Used
-----------------
1. Class
2. Object
3. Constructor (__init__)
4. Encapsulation

===========================================================
"""


class Customer:

    def __init__(self, customer_name):
        """
        Constructor

        __init__() is automatically called whenever an object
        is created.

        It initializes the object variables.
        """

        self.customer_name = customer_name

**Note:** This class currently stores just `customer_name`, but the pattern is deliberately extensible — a real system would likely add `phone_number` or `email` here later without touching any other file, exactly because `Customer` is isolated from `MovieReservation`'s booking logic.

## 2. 🔢 `booking.py` — The `Booking` Class

Demonstrates a **class variable** used as a shared, auto-incrementing counter. Every call to `generate_booking_id()` bumps the counter for *all* future bookings, guaranteeing unique IDs like `BK1001`, `BK1002`, `BK1003`, ...

In [ ]:
"""
===========================================================
File Name : booking.py

THEORY - CLASS VARIABLE

A Class Variable belongs to the class rather than individual
objects.

It is shared among all objects.

Example

Booking Counter

Customer 1 -> BK1001

Customer 2 -> BK1002

Customer 3 -> BK1003

All objects share the same counter.

===========================================================
"""


class Booking:

    booking_counter = 1001

    @classmethod
    def generate_booking_id(cls):
        """
        Class Method

        A class method works with class variables.

        cls refers to the class itself.
        """

        booking_id = "BK" + str(cls.booking_counter)

        cls.booking_counter += 1

        return booking_id
    

**Note:** `generate_booking_id` is a `@classmethod`, so it receives `cls` (the class itself) instead of `self` (an instance) — that's necessary because `booking_counter` belongs to the *class*, not to any particular `Booking` object (in fact, this code never even creates a `Booking()` instance — it only ever calls the class method directly).

Let's verify this actually behaves as claimed — each call should return a new, incrementing ID:

In [ ]:
# Verifying Booking.generate_booking_id() increments correctly
print(Booking.generate_booking_id())  # BK1001
print(Booking.generate_booking_id())  # BK1002
print(Booking.generate_booking_id())  # BK1003
print("Counter is now:", Booking.booking_counter)


**Verified:** each call returns a fresh, incrementing ID and `Booking.booking_counter` keeps climbing — confirmed by actually running this cell.

## 3. 🎬 `movie.py` — The `MovieReservation` Class (Encapsulation)

This is the heart of the project: `MovieReservation` bundles `movie_name`, `total_seats`, and `booked_seats` together with all the methods that read or modify them (`book_ticket`, `cancel_ticket`, `show_available_seats`, `show_bookings`) — nothing outside this class needs to know that seats are stored internally as a dictionary keyed by seat number.

In [ ]:
"""
===========================================================
File Name : movie.py

THEORY - ENCAPSULATION

Encapsulation means binding data and methods together
inside a class.

The class controls how the data is accessed.

Here,

Data
----
movie_name
total_seats
booked_seats

Methods
-------
book_ticket()
cancel_ticket()
show_available_seats()
show_bookings()

Everything is packed together inside MovieReservation class.

===========================================================
"""

# NOTE: in the original 4-file project these came from:
#   from PythonTutorial.oops.book_reservation.booking import Booking
#   from PythonTutorial.oops.book_reservation.customer import Customer
# Those imports only work when the files actually live in that package on disk.
# Since Booking and Customer are already defined above in this notebook
# (Sections 1 and 2), no import is needed here - they're already in scope.


class MovieReservation:

    def __init__(self, movie_name, total_seats):
        """
        Constructor

        Initializes movie details.
        """

        self.movie_name = movie_name
        self.total_seats = total_seats
        self.booked_seats = {}

    def show_available_seats(self):

        print("\nAvailable Seats")

        available = []

        for seat in range(1, self.total_seats + 1):

            if seat not in self.booked_seats:
                available.append(seat)

        print(available)

    def book_ticket(self, customer_name, seat_number):

        """
        Validation

        Prevent duplicate bookings.
        """

        if seat_number < 1 or seat_number > self.total_seats:
            print("Invalid Seat Number")
            return

        if seat_number in self.booked_seats:
            print("Seat Already Booked")
            return

        customer = Customer(customer_name)

        booking_id = Booking.generate_booking_id()

        self.booked_seats[seat_number] = {
            "Booking ID": booking_id,
            "Customer": customer.customer_name,
            "Seat": seat_number
        }

        print("\nBooking Successful")
        print("Booking ID :", booking_id)
        print("Customer :", customer.customer_name)
        print("Seat :", seat_number)

    def cancel_ticket(self, booking_id):

        for seat, details in list(self.booked_seats.items()):

            if details["Booking ID"] == booking_id:

                del self.booked_seats[seat]

                print("Booking Cancelled Successfully")
                return

        print("Booking ID Not Found")

    def show_bookings(self):

        if not self.booked_seats:

            print("No Bookings")
            return

        print("\nBooked Tickets")

        for booking in self.booked_seats.values():

            print("----------------------------")
            print("Booking ID :", booking["Booking ID"])
            print("Customer :", booking["Customer"])
            print("Seat :", booking["Seat"])


**Bug fixed:** the original cell had `from PythonTutorial.oops.book_reservation.booking import Booking` and a matching import for `Customer`. Those only resolve when the four files actually sit inside a real `PythonTutorial/oops/book_reservation/` package on disk with proper `__init__.py` files — pasted into a notebook, `ModuleNotFoundError: No module named 'PythonTutorial'` is raised immediately (confirmed by running it). Since `Booking` and `Customer` are already defined above in this same notebook, the fix simply removes the imports and relies on the classes already being in scope — this is *only* valid for the notebook's single shared namespace, not a real multi-file project.

**Note:** `book_ticket` creates a **brand-new** `Customer` object every single call, even if the same person books two different seats — nothing links repeat customers together. A more complete design might look up an existing `Customer` by name first. Also worth noting: `cancel_ticket` loops through *every* booking checking `details["Booking ID"] == booking_id` — with a dict keyed by seat number instead of booking ID, this lookup is O(n) rather than O(1); keying `booked_seats` by booking ID (or keeping a second index) would make cancellation faster for large numbers of bookings.

### Let's actually run the reservation flow (non-interactively)

The real `main.py` below is interactive (`input()`-driven), which can't run automatically in an execution check. Here's the same sequence of calls with hardcoded values instead of `input()`, to verify `MovieReservation` behaves as documented.

In [ ]:
movie = MovieReservation("Avengers Endgame", 10)

movie.show_available_seats()
movie.book_ticket("Pratham", 5)
movie.book_ticket("Isha", 5)     # duplicate seat -> should be rejected
movie.book_ticket("Anshu", 20)   # invalid seat number -> should be rejected
movie.show_bookings()
movie.cancel_ticket("BK1004")    # booking id from Section 2's demo run above
movie.show_bookings()
movie.cancel_ticket("BK9999")    # doesn't exist -> should say "Not Found"


**Verified:** duplicate-seat booking is correctly rejected, invalid seat numbers are correctly rejected, cancellation removes the right booking, and cancelling a non-existent booking ID correctly reports "Booking ID Not Found" — confirmed by actually running this cell. (Note the booking ID printed starts from wherever `Booking.booking_counter` left off after Section 2's demo, since it's one shared counter across this whole notebook.)

## 4. 🖥️ `main.py` — The Driver Program

`main.py` is the **Driver Program**. It is responsible for:
1. Creating objects
2. Calling methods
3. Taking user input
4. Running the program

All the real business logic stays inside the classes above — this follows the OOP principle of **Separation of Concerns**. This cell is interactive (it calls `input()` in a loop) and is shown here for reference; run it directly if you want to try the full menu.

In [ ]:
"""
===========================================================
File Name : main.py

THEORY

main.py is called the Driver Program.

It is responsible for

1. Creating Objects

2. Calling Methods

3. Taking User Input

4. Running the Program

The business logic remains inside the classes.

This follows the OOP principle of Separation of Concerns.

===========================================================
"""

# NOTE: originally `from PythonTutorial.oops.book_reservation.movie import MovieReservation`
# Removed here since MovieReservation is already defined above in this notebook (Section 3).

movie = MovieReservation("Avengers Endgame", 10)

while True:

    print("\n========== MOVIE RESERVATION ==========")
    print("1. Show Available Seats")
    print("2. Book Ticket")
    print("3. Cancel Ticket")
    print("4. Show Bookings")
    print("5. Exit")

    choice = input("Enter Choice : ")

    if choice == "1":

        movie.show_available_seats()

    elif choice == "2":

        name = input("Customer Name : ")

        seat = int(input("Seat Number : "))

        movie.book_ticket(name, seat)

    elif choice == "3":

        booking_id = input("Booking ID : ")

        movie.cancel_ticket(booking_id)

    elif choice == "4":

        movie.show_bookings()

    elif choice == "5":

        print("Thank You")
        break

    else:

        print("Invalid Choice")


**Bug fixed:** same cross-package import issue as `movie.py` — `from PythonTutorial.oops.book_reservation.movie import MovieReservation` doesn't resolve inside a notebook and raises `ModuleNotFoundError`. Removed since `MovieReservation` is already defined above.

**Note:** This cell is **interactive** — running it starts a loop that calls `input()` repeatedly and only stops when you enter `5`. The non-interactive demo in Section 3 above exercises the exact same underlying methods (`show_available_seats`, `book_ticket`, `cancel_ticket`, `show_bookings`) without needing manual input, which is how this notebook verifies the logic actually works.

## ⚠ Common Misconceptions

❌ Code split across multiple `.py` files can be pasted into one notebook unchanged.
✅ Cross-file `import` statements (like `from PythonTutorial.oops.book_reservation.booking import Booking`) only work when those files exist in a real package structure on disk. Inside a notebook where everything runs in one shared namespace, those imports must be removed — the classes are already available once their defining cell has run.

❌ A "Driver Program" (`main.py`) should contain business logic.
✅ Its job is only to create objects and call their methods (often driven by user input). Validation and state changes belong inside the classes (`MovieReservation`, etc.) — that's Separation of Concerns.

❌ `@classmethod` and `@staticmethod` are interchangeable.
✅ `@classmethod` receives `cls` (the class) and can read/modify class variables like `booking_counter`; `@staticmethod` receives neither `self` nor `cls` and can't touch class or instance state at all.

## 🔍 Interview Questions

- Why does `movie.py` need to import `Booking` and `Customer` in the original multi-file project, but not when everything's pasted into one notebook?
- What is "Separation of Concerns," and how does splitting this project into `main.py` / `movie.py` / `customer.py` / `booking.py` demonstrate it?
- Why is `generate_booking_id` a `@classmethod` instead of a regular instance method or a `@staticmethod`?
- What would happen to Booking IDs if `booking_counter` were an *instance* variable instead of a class variable?
- How would you change `MovieReservation` to avoid creating a brand-new `Customer` object every time the same person books another seat?
- `cancel_ticket` loops through every booking to find a matching Booking ID — how could you restructure `booked_seats` to make that lookup O(1) instead?

## 🎯 Key Takeaways

1. Real projects split responsibilities across files/modules (`Customer`, `Booking`, `MovieReservation`, and a driver `main.py`) instead of cramming everything into one place.
2. `@classmethod` is the right tool when a method needs to read/modify a **class-level** variable (like a shared counter) rather than any single object's data.
3. A class variable used as a counter (`Booking.booking_counter`) persists and increments across every call, guaranteeing unique IDs without needing a database.
4. `main.py`'s job is orchestration only — creating objects, taking input, calling methods — while validation and state changes stay inside the classes (Separation of Concerns).
5. Cross-file imports are a real-project concept; when consolidating multi-file code into a single notebook for study purposes, those imports need to be stripped since everything already shares one namespace.